# ANALIZA DANYCH Z ZEGARKA APPLE WATCH


## Podstawowe informacje o pliku:

* Plik jest pobierany z aplikacji Zdrowie (Apple Health) na telefonie
* format: XML
* struktura: hierarchiczna, z wieloma różnymi typami rekordów (Record, Correlation, Workout, ActivitySummary, ClinicalRecord, Audiogram, VisionPrescription)
* zawiera dane dotyczące zdrowia, aktywności, snu, tętna, kroków, itp.
* jest dość duży ponad 700 mb





## TO DO:
1. [x] rozgryźć strukturę pliku xml - pól w nim zawartych - co dokładnie w nim jest
2. [x] sprawdzić pomiar EKG i możliwość ściągnięcia danych
3. [x] napisać kod pythona do wczytania tych danych
4. [x] zastanowić się nad programem ćwiczeń dla osób o różnej kondycji, z pomiarem zegarkiem

In [9]:
import os
import streamlit as st
import time

from altair.datasets import data
from lxml import etree as ET
import pandas as pd
from parser import load_one_type, load_all_health_data


In [10]:
path = "06-04/apple_health_export/eksport.xml"

In [11]:
def discover_xml_structure(file_path):
    # Słownik: Klucz = Nazwa Tagu, Wartość = Zbiór atrybutów 'type' lub 'workoutActivityType'
    found_structures = {}

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        tag_name = elem.tag

        #  dodajemy do slownika jesli nie ma w nim jeszcze
        if tag_name not in found_structures:
            found_structures[tag_name] = set()

        # sprawdzamy typy głównych tagów
        sub_type = elem.get("type") or elem.get("workoutActivityType") or "Brak"

        found_structures[tag_name].add(sub_type)

        # Czyścimy RAM
        elem.clear()

    return found_structures

moje_dane = discover_xml_structure(path)

print("ODKRYTE TAGI + typy danych")
for tag, sub_types in moje_dane.items():
    print(f"\n TAG: <{tag}>")
    print(f"   Liczba różnych typów: {len(sub_types)}")
    for st in list(sub_types):
        print(f"     - {st}")

ODKRYTE TAGI + typy danych

 TAG: <ExportDate>
   Liczba różnych typów: 1
     - Brak

 TAG: <Me>
   Liczba różnych typów: 1
     - Brak

 TAG: <Record>
   Liczba różnych typów: 41
     - HKQuantityTypeIdentifierRespiratoryRate
     - HKQuantityTypeIdentifierWalkingDoubleSupportPercentage
     - HKCategoryTypeIdentifierSleepAnalysis
     - HKQuantityTypeIdentifierHeadphoneAudioExposure
     - HKQuantityTypeIdentifierBodyFatPercentage
     - HKQuantityTypeIdentifierEnvironmentalSoundReduction
     - HKQuantityTypeIdentifierStairDescentSpeed
     - HKQuantityTypeIdentifierOxygenSaturation
     - HKQuantityTypeIdentifierAppleExerciseTime
     - HKQuantityTypeIdentifierWalkingAsymmetryPercentage
     - HKDataTypeSleepDurationGoal
     - HKQuantityTypeIdentifierHeight
     - HKQuantityTypeIdentifierAppleWalkingSteadiness
     - HKQuantityTypeIdentifierAppleSleepingWristTemperature
     - HKQuantityTypeIdentifierHeartRateRecoveryOneMinute
     - HKCategoryTypeIdentifierAudioExposureEvent
   

In [12]:
important_tags = {
    'Me',"Record", "Workout", "ActivitySummary", "InstantaneousBeatsPerMinute"
}


In [13]:
print(moje_dane)

{'ExportDate': {'Brak'}, 'Me': {'Brak'}, 'Record': {'HKQuantityTypeIdentifierRespiratoryRate', 'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierWalkingAsymmetryPercentage', 'HKDataTypeSleepDurationGoal', 'HKQuantityTypeIdentifierHeight', 'HKQuantityTypeIdentifierAppleWalkingSteadiness', 'HKQuantityTypeIdentifierAppleSleepingWristTemperature', 'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute', 'HKCategoryTypeIdentifierAudioExposureEvent', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKQuantityTypeIdentifierEnvironmentalAudioExposure', 'HKQuantityTypeIdentifierDistanceCycling', 'HKQuantityTypeIdentifierSixMinuteWalkTestDistan

In [14]:

diff_types = set()

diff_tags = set()
# Iterujemy po pliku
for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag in important_tags:
        typ = elem.get("type")  or elem.get("workoutActivityType") or elem.tag

        if typ not in diff_types:
            print(f"\n TYP: {typ}")
            print(f"   Atrybuty: {elem.attrib}")


            children = list(elem)
            if children:
                for child in children:
                    print(f"   typ: <{child.tag}> atrybuty: {child.attrib}")
            diff_types.add(typ)



        elem.clear()




 TYP: Me
   Atrybuty: {'HKCharacteristicTypeIdentifierDateOfBirth': '2006-03-02', 'HKCharacteristicTypeIdentifierBiologicalSex': 'HKBiologicalSexFemale', 'HKCharacteristicTypeIdentifierBloodType': 'HKBloodTypeNotSet', 'HKCharacteristicTypeIdentifierFitzpatrickSkinType': 'HKFitzpatrickSkinTypeNotSet', 'HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse': 'Brak'}

 TYP: HKQuantityTypeIdentifierHeight
   Atrybuty: {'type': 'HKQuantityTypeIdentifierHeight', 'sourceName': 'FitnessOnline', 'sourceVersion': '206', 'unit': 'cm', 'creationDate': '2025-08-12 12:58:06 +0200', 'startDate': '2025-08-12 12:58:05 +0200', 'endDate': '2025-08-12 12:58:05 +0200', 'value': '163'}

 TYP: HKQuantityTypeIdentifierBodyMass
   Atrybuty: {'type': 'HKQuantityTypeIdentifierBodyMass', 'sourceName': 'Zdrowie', 'sourceVersion': '14.7.1', 'unit': 'kg', 'creationDate': '2021-09-06 11:46:43 +0200', 'startDate': '2021-09-06 11:46:00 +0200', 'endDate': '2021-09-06 11:46:00 +0200', 'value': '55'}
   typ: <Metadat

In [15]:
print(diff_types)
print(len(diff_types))

{'HKQuantityTypeIdentifierRespiratoryRate', 'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKWorkoutActivityTypePilates', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKWorkoutActivityTypeCycling', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierWalkingAsymmetryPercentage', 'HKDataTypeSleepDurationGoal', 'HKQuantityTypeIdentifierHeight', 'ActivitySummary', 'HKQuantityTypeIdentifierAppleWalkingSteadiness', 'HKQuantityTypeIdentifierAppleSleepingWristTemperature', 'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute', 'HKCategoryTypeIdentifierAudioExposureEvent', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKQuantityTypeIdentifierEnvironmentalAudioExposure', 'HKQuantityTypeIdentifierDistanceCycling', 'HKQuantityTypeI

In [16]:
#Stworzenie dataframe z wybranym typem danych

short_names = {
    # --- CIAŁO / PODSTAWOWE ---
    'HKQuantityTypeIdentifierHeight': 'height',
    'HKQuantityTypeIdentifierBodyMass': 'weight',
    'HKQuantityTypeIdentifierWaistCircumference': 'waist',
    'HKQuantityTypeIdentifierBodyFatPercentage': 'fat_pct',

    # --- SERCE (HEART) ---
    'HKQuantityTypeIdentifierHeartRate': 'hr',
    'HKQuantityTypeIdentifierRestingHeartRate': 'rhr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage': 'hr_walk_avg',
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute': 'hrr_1min',
    'HKCategoryTypeIdentifierHighHeartRateEvent': 'hr_high_event',

    # --- AKTYWNOŚĆ / ENERGIA ---
    'HKQuantityTypeIdentifierStepCount': 'steps',
    'HKQuantityTypeIdentifierDistanceWalkingRunning': 'dist_walk',
    'HKQuantityTypeIdentifierDistanceCycling': 'dist_cycle',
    'HKQuantityTypeIdentifierBasalEnergyBurned': 'kcal_basal',
    'HKQuantityTypeIdentifierActiveEnergyBurned': 'kcal_active',
    'HKQuantityTypeIdentifierFlightsClimbed': 'flights',
    'HKQuantityTypeIdentifierAppleExerciseTime': 'exercise_min',
    'HKQuantityTypeIdentifierAppleStandTime': 'stand_min',
    'HKCategoryTypeIdentifierAppleStandHour': 'stand_hr',
    'HKQuantityTypeIdentifierPhysicalEffort': 'effort',
    'HKQuantityTypeIdentifierVO2Max': 'vo2max',

    # --- MOBILNOŚĆ / CHÓD ---
    'HKQuantityTypeIdentifierWalkingSpeed': 'walk_speed',
    'HKQuantityTypeIdentifierWalkingStepLength': 'walk_step_len',
    'HKQuantityTypeIdentifierWalkingAsymmetryPercentage': 'walk_asym',
    'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage': 'walk_dbl_supp',
    'HKQuantityTypeIdentifierStairAscentSpeed': 'stair_up_speed',
    'HKQuantityTypeIdentifierStairDescentSpeed': 'stair_down_speed',
    'HKQuantityTypeIdentifierAppleWalkingSteadiness': 'walk_steadiness',
    'HKQuantityTypeIdentifierSixMinuteWalkTestDistance': 'walk_6min',

    # --- ODDECH / POZIOMY ---
    'HKQuantityTypeIdentifierOxygenSaturation': 'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate': 'resp_rate',

    # --- SEN ---
    'HKCategoryTypeIdentifierSleepAnalysis': 'sleep',
    'HKDataTypeSleepDurationGoal': 'sleep_goal',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',

    # --- ŚRODOWISKO / DŹWIĘK ---
    'HKQuantityTypeIdentifierEnvironmentalAudioExposure': 'audio_env',
    'HKQuantityTypeIdentifierHeadphoneAudioExposure': 'audio_hp',
    'HKQuantityTypeIdentifierEnvironmentalSoundReduction': 'sound_red',
    'HKCategoryTypeIdentifierAudioExposureEvent': 'audio_event',
    'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent': 'audio_hp_event',
    'HKQuantityTypeIdentifierTimeInDaylight': 'daylight',

    # --- INNE ---
    'HKCategoryTypeIdentifierMenstrualFlow': 'menstr'
}


In [17]:
def load_one_type(file_path, tag_name, type_name=None):
    """
    Pobiera dane na podstawie samego tagu (np. 'ActivitySummary') lub tagu i typu (np. 'Record' + 'hr').
    Chirurgiczna pęseta do eksploracji danych.
    """
    health_data = []

    if file_path is None:
        return "Zła ścieżka pliku"

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:

        if elem.tag == tag_name:
            current_type = elem.get('type') or elem.get('workoutActivityType')
            if type_name is None or current_type == type_name:
                record = dict(elem.attrib)

                for child in elem:
                    if child.tag == "MetadataEntry":
                        key = child.get('key')
                        if key is not None:
                            record[key] = child.get('value')
                health_data.append(record)

            elem.clear()

    if health_data:
        df = pd.DataFrame(health_data)
        if 'value' in df.columns:
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
        for date_col in ['startDate', 'endDate', 'creationDate']:
            if date_col in df.columns:
                df[date_col] = pd.to_datetime(df[date_col])
        return df
    return pd.DataFrame()


df_hr = load_one_type(path, 'Record', 'HKQuantityTypeIdentifierHeartRate')
display(df_hr.head())


,type,sourceName,sourceVersion,unit,creationDate,startDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext,HKMetadataKeySyncVersion,HKMetadataKeySyncIdentifier
0,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 17:29:14+02:00,2021-12-17 01:00:00+02:00,2021-12-17 01:00:59+02:00,72.0,NaN,NaN,NaN,NaN
1,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 17:29:14+02:00,2021-12-17 01:01:00+02:00,2021-12-17 01:01:59+02:00,70.0,NaN,NaN,NaN,NaN
2,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 17:29:14+02:00,2021-12-17 01:02:00+02:00,2021-12-17 01:02:59+02:00,70.0,NaN,NaN,NaN,NaN
3,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 17:29:14+02:00,2021-12-17 01:03:00+02:00,2021-12-17 01:03:59+02:00,69.0,NaN,NaN,NaN,NaN
4,HKQuantityTypeIdentifierHeartRate,Zepp Life,202112101619,count/min,2021-12-24 17:29:14+02:00,2021-12-17 01:04:00+02:00,2021-12-17 01:04:59+02:00,70.0,NaN,NaN,NaN,NaN


Analiza samych danych tetna


In [18]:
print(df_hr.columns)
print(df_hr.head())
print(df_hr.describe())


Index(['type', 'sourceName', 'sourceVersion', 'unit', 'creationDate',
       'startDate', 'endDate', 'value', 'device',
       'HKMetadataKeyHeartRateMotionContext', 'HKMetadataKeySyncVersion',
       'HKMetadataKeySyncIdentifier'],
      dtype='object')
                                type sourceName sourceVersion       unit  \
0  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
1  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
2  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
3  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   
4  HKQuantityTypeIdentifierHeartRate  Zepp Life  202112101619  count/min   

               creationDate                 startDate  \
0 2021-12-24 17:29:14+02:00 2021-12-17 01:00:00+02:00   
1 2021-12-24 17:29:14+02:00 2021-12-17 01:01:00+02:00   
2 2021-12-24 17:29:14+02:00 2021-12-17 01:02:00+02:00   
3 2021-12-24 17:29:14+02:00 2021-12-17 01:03:00+02:00   
4 2

Czyścimy  dane - usuwamy kolumny w których wszystkie wartości sa nan

In [19]:
df_hr = df_hr.dropna(axis=1, how='all')

In [20]:
# Usuwamy konkretne kolumny techniczne, których nie potrzebujemy do analizy tętna
df_hr = df_hr.drop(columns=['HKMetadataKeySyncVersion', 'HKMetadataKeySyncIdentifier'], errors='ignore')

In [21]:
print(df_hr.columns)

Index(['type', 'sourceName', 'sourceVersion', 'unit', 'creationDate',
       'startDate', 'endDate', 'value', 'device',
       'HKMetadataKeyHeartRateMotionContext'],
      dtype='object')


In [22]:
# Usuwamy rekordy, które mają IDENTYCZNY start, koniec I wartość
df_hr = df_hr.drop_duplicates(subset=['startDate', 'endDate', 'value'])

print(f"Usunięto {len(df_hr) - len(df_hr)} idealnych duplikatów.")

Usunięto 0 idealnych duplikatów.


In [23]:

# bierzemy dane pobierane tylko z zegarka
df = df_hr[df_hr['sourceName'].str.contains('Apple', na=False)]


# Wywalamy rekordy, gdzie koniec jest przed początkiem i jakieś błędne
df = df[df['endDate'] >= df['startDate']]

# indeksujemy po dacie poczatkowej
df = df.set_index('startDate')




In [24]:
display(df.tail(100))

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2026-04-05 21:27:10+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 21:27:14+02:00,2026-04-05 21:27:10+02:00,143.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",2
2026-04-05 21:27:18+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 21:27:19+02:00,2026-04-05 21:27:18+02:00,147.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",2
2026-04-05 21:27:23+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 21:27:24+02:00,2026-04-05 21:27:23+02:00,146.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",2
2026-04-05 21:27:26+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 21:27:29+02:00,2026-04-05 21:27:26+02:00,146.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",2
2026-04-05 21:27:32+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 21:27:34+02:00,2026-04-05 21:27:32+02:00,144.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",2
...,...,...,...,...,...,...,...,...,...
2026-04-05 22:31:51+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:37:52+02:00,2026-04-05 22:31:51+02:00,76.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:37:49+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:43:35+02:00,2026-04-05 22:37:49+02:00,74.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:43:16+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:45:34+02:00,2026-04-05 22:43:16+02:00,72.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",0


In [25]:

# 2. Definiujemy reguły: dla 'value' średnia, dla reszty 'pierwszy element'
# Tworzymy słownik, który dla każdej kolumny przypisuje funkcję 'first'
agg_rules = {col: 'first' for col in df.columns}

# Nadpisujemy regułę dla kolumny 'value' na 'mean'
if 'value' in agg_rules:
    agg_rules['value'] = 'mean'

# 3. Magiczne resamplowanie z regułami
df_minutowe = df.resample('1min').agg(agg_rules)

# 4. Przywracamy indeks do kolumny
df_minutowe = df_minutowe.reset_index()

display(df_minutowe)

,startDate,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
0,2024-03-03 17:10:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:10:31+02:00,2024-03-03 17:10:22+02:00,93.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
1,2024-03-03 17:11:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
2,2024-03-03 17:12:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:16:07+02:00,2024-03-03 17:12:16+02:00,84.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
3,2024-03-03 17:13:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
4,2024-03-03 17:14:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...
1099057,2026-04-05 22:47:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
1099058,2026-04-05 22:48:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
1099059,2026-04-05 22:49:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None
1099060,2026-04-05 22:50:00+02:00,None,None,None,None,NaT,NaT,NaN,None,None


In [27]:
df = df_minutowe.dropna()
display(df)

,startDate,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
0,2024-03-03 17:10:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:10:31+02:00,2024-03-03 17:10:22+02:00,93.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2,2024-03-03 17:12:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:16:07+02:00,2024-03-03 17:12:16+02:00,84.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
8,2024-03-03 17:18:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:21:08+02:00,2024-03-03 17:18:43+02:00,82.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
14,2024-03-03 17:24:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:25:36+02:00,2024-03-03 17:24:23+02:00,80.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
15,2024-03-03 17:25:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:30:49+02:00,2024-03-03 17:25:28+02:00,78.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
...,...,...,...,...,...,...,...,...,...,...
1099041,2026-04-05 22:31:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:37:52+02:00,2026-04-05 22:31:51+02:00,76.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
1099047,2026-04-05 22:37:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:43:35+02:00,2026-04-05 22:37:49+02:00,74.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
1099053,2026-04-05 22:43:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:45:34+02:00,2026-04-05 22:43:16+02:00,72.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",0
1099055,2026-04-05 22:45:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:51:13+02:00,2026-04-05 22:45:26+02:00,76.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1


In [28]:

df.set_index('startDate', inplace=True)
df.sort_index(inplace=True)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 177180 entries, 2024-03-03 17:10:00+02:00 to 2026-04-05 22:51:00+02:00
Data columns (total 9 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   type                                 177180 non-null  object                   
 1   sourceName                           177180 non-null  object                   
 2   sourceVersion                        177180 non-null  object                   
 3   unit                                 177180 non-null  object                   
 4   creationDate                         177180 non-null  datetime64[ns, UTC+02:00]
 5   endDate                              177180 non-null  datetime64[ns, UTC+02:00]
 6   value                                177180 non-null  float64                  
 7   device                               177180 non-null  object              

In [30]:
df = df.dropna(subset=['value'])
print(df.info())
df

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 177180 entries, 2024-03-03 17:10:00+02:00 to 2026-04-05 22:51:00+02:00
Data columns (total 9 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   type                                 177180 non-null  object                   
 1   sourceName                           177180 non-null  object                   
 2   sourceVersion                        177180 non-null  object                   
 3   unit                                 177180 non-null  object                   
 4   creationDate                         177180 non-null  datetime64[ns, UTC+02:00]
 5   endDate                              177180 non-null  datetime64[ns, UTC+02:00]
 6   value                                177180 non-null  float64                  
 7   device                               177180 non-null  object              

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2024-03-03 17:10:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:10:31+02:00,2024-03-03 17:10:22+02:00,93.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:12:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:16:07+02:00,2024-03-03 17:12:16+02:00,84.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:18:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:21:08+02:00,2024-03-03 17:18:43+02:00,82.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:24:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:25:36+02:00,2024-03-03 17:24:23+02:00,80.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:25:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:30:49+02:00,2024-03-03 17:25:28+02:00,78.0,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
...,...,...,...,...,...,...,...,...,...
2026-04-05 22:31:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:37:52+02:00,2026-04-05 22:31:51+02:00,76.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:37:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:43:35+02:00,2026-04-05 22:37:49+02:00,74.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:43:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:45:34+02:00,2026-04-05 22:43:16+02:00,72.0,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",0


In [31]:
df.loc['2026-3-3':]

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2026-03-03 00:00:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:04:23+02:00,2026-03-03 00:00:38+02:00,67.0000,"<<HKDevice: 0xab1076c10>, name:Apple Watch, ma...",0
2026-03-03 00:01:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:02:14+02:00,2026-03-03 00:01:12+02:00,75.5206,"<<HKDevice: 0xab1076c10>, name:Apple Watch, ma...",0
2026-03-03 00:04:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:09:28+02:00,2026-03-03 00:04:00+02:00,68.0000,"<<HKDevice: 0xab1076c10>, name:Apple Watch, ma...",0
2026-03-03 00:09:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:15:14+02:00,2026-03-03 00:09:12+02:00,74.0000,"<<HKDevice: 0xab1076c10>, name:Apple Watch, ma...",0
2026-03-03 00:14:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-03-03 00:56:07+02:00,2026-03-03 00:14:32+02:00,90.0000,"<<HKDevice: 0xab1076c10>, name:Apple Watch, ma...",0
...,...,...,...,...,...,...,...,...,...
2026-04-05 22:31:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:37:52+02:00,2026-04-05 22:31:51+02:00,76.0000,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:37:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:43:35+02:00,2026-04-05 22:37:49+02:00,74.0000,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",1
2026-04-05 22:43:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),11.6.2,count/min,2026-04-05 22:45:34+02:00,2026-04-05 22:43:16+02:00,72.0000,"<<HKDevice: 0xab1023a70>, name:Apple Watch, ma...",0


In [32]:
df.loc['2024-03-01':'2024-03-05']

,type,sourceName,sourceVersion,unit,creationDate,endDate,value,device,HKMetadataKeyHeartRateMotionContext
startDate,,,,,,,,,
2024-03-03 17:10:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:10:31+02:00,2024-03-03 17:10:22+02:00,93.000,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:12:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:16:07+02:00,2024-03-03 17:12:16+02:00,84.000,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:18:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:21:08+02:00,2024-03-03 17:18:43+02:00,82.000,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:24:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:25:36+02:00,2024-03-03 17:24:23+02:00,80.000,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
2024-03-03 17:25:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3,count/min,2024-03-03 17:30:49+02:00,2024-03-03 17:25:28+02:00,78.000,"<<HKDevice: 0xab0f57e30>, name:Apple Watch, ma...",1
...,...,...,...,...,...,...,...,...,...
2024-03-05 23:41:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3.1,count/min,2024-03-05 23:45:52+02:00,2024-03-05 23:41:47+02:00,67.000,"<<HKDevice: 0xab1092e40>, name:Apple Watch, ma...",0
2024-03-05 23:47:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3.1,count/min,2024-03-05 23:48:27+02:00,2024-03-05 23:47:57+02:00,65.000,"<<HKDevice: 0xab1092e40>, name:Apple Watch, ma...",1
2024-03-05 23:50:00+02:00,HKQuantityTypeIdentifierHeartRate,Apple Watch (Lol),10.3.1,count/min,2024-03-05 23:55:34+02:00,2024-03-05 23:50:10+02:00,63.000,"<<HKDevice: 0xab1092e40>, name:Apple Watch, ma...",1


# Analiza danych EKG
niestety nie ma ich w pliku XML, ale są w oddzielnym pliku CSV, który można pobrać z aplikacji Zdrowie. Zawiera próbki napięcia zarejestrowane podczas pomiaru EKG. Każdy wiersz reprezentuje jedną próbkę, z kolumnami 'sample' (numer próbki) i 'voltage' (napięcie w mV). Dane te są przydatne do analizy rytmu serca i wykrywania ewentualnych arytmii.

In [33]:
ekg_data = pd.read_csv("ekg.csv", skiprows=12, names=['sample', 'voltage'])